## Celda 1 — Importaciones
---

In [1]:
import os, json, warnings
warnings.filterwarnings("ignore")

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType

from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import (
    LinearRegression, RandomForestRegressor
)
from pyspark.ml.classification import (
    LogisticRegression, RandomForestClassifier
)
from pyspark.ml.evaluation import (
    RegressionEvaluator, BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator
)

spark = (
    SparkSession.builder
    .appName("wrangler_temperatura_02")
    .config("spark.sql.session.timeZone", "America/Lima")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark listo:", spark.version)

Spark listo: 3.5.3


## Celda 2 — Rutas y cargar plata + umbrales
---

In [2]:
# Rutas
SILVER  = "/home/jovyan/work/data/silver/wrangler_temperatura"
MODELS  = "/home/jovyan/work/models/wrangler_temperatura"

# Cargar plata
df = spark.read.parquet(SILVER).orderBy("minuto")
total = df.count()
print(f"Filas en plata: {total:,}")

# Cargar umbrales
with open(f"{MODELS}/umbrales.json") as f:
    umbrales = json.load(f)

P5  = umbrales["p5"]
P95 = umbrales["p95"]
print(f"P5  = {P5} °C")
print(f"P95 = {P95} °C")

Filas en plata: 4,862
P5  = 16.5336 °C
P95 = 26.6225 °C


## Celda 3 — Construir rezagos por unión de tiempo
---

In [3]:
# Alias base
base = df.select(
    F.col("minuto"),
    F.col("temperatura_c"),
    F.col("humedad_pct"),
    F.col("presion_hpa"),
)

# Función para crear rezago uniendo por tiempo
def agregar_rezago(df_base, df_ref, minutos, col="temperatura_c"):
    alias = f"lag{minutos}"
    df_ref2 = df_ref.select(
        F.col("minuto").alias(f"minuto_{alias}"),
        F.col(col).alias(f"temp_{alias}")
    )
    return df_base.join(
        df_ref2,
        df_base["minuto"] == (
            F.col(f"minuto_{alias}") + F.expr(f"INTERVAL {minutos} MINUTES")
        ),
        how="left"
    ).drop(f"minuto_{alias}")

# Agregar rezagos t-15, t-30, t-60
df_lags = agregar_rezago(base, base, 15)
df_lags = agregar_rezago(df_lags, base, 30)
df_lags = agregar_rezago(df_lags, base, 60)

# Cambio reciente (temperatura actual menos la de hace 15 min)
df_lags = df_lags.withColumn(
    "delta_15",
    F.col("temperatura_c") - F.col("temp_lag15")
)

# Hora en forma cíclica
df_lags = df_lags.withColumn(
    "hora_sin", F.sin(2 * 3.14159 * F.hour("minuto") / 24)
).withColumn(
    "hora_cos", F.cos(2 * 3.14159 * F.hour("minuto") / 24)
)

# Descartar filas donde falta algún rezago
df_feat = df_lags.dropna(subset=["temp_lag15", "temp_lag30", "temp_lag60"])

print(f"Filas con todos los rezagos: {df_feat.count():,}")
df_feat.show(3, truncate=False)

Filas con todos los rezagos: 3,622
+-------------------+-------------+-----------+-----------+----------+----------+----------+--------------------+--------+--------+
|minuto             |temperatura_c|humedad_pct|presion_hpa|temp_lag15|temp_lag30|temp_lag60|delta_15            |hora_sin|hora_cos|
+-------------------+-------------+-----------+-----------+----------+----------+----------+--------------------+--------+--------+
|2026-10-22 00:52:00|16.9358      |37.7427    |646.7      |17.084    |17.1228   |18.7746   |-0.14819999999999922|0.0     |1.0     |
|2026-10-22 00:53:00|16.9352      |38.5161    |646.66     |17.069    |17.1167   |18.7791   |-0.1338000000000008 |0.0     |1.0     |
|2026-10-22 00:54:00|16.9198      |37.827     |646.65     |17.0799   |17.0994   |18.8374   |-0.1600999999999999 |0.0     |1.0     |
+-------------------+-------------+-----------+-----------+----------+----------+----------+--------------------+--------+--------+
only showing top 3 rows



## Celda 4 — Corte cronológico 80/20
---

In [4]:
# Ordenar y obtener el timestamp de corte
from pyspark.sql import Window

total_feat = df_feat.count()
n_train = int(total_feat * 0.80)

# Obtener el minuto en la posición n_train
df_ordenado = df_feat.orderBy("minuto")
minuto_corte = df_ordenado.select("minuto").collect()[n_train]["minuto"]

print(f"Total filas con rezagos: {total_feat:,}")
print(f"Filas entrenamiento (80%): {n_train:,}")
print(f"Minuto de corte: {minuto_corte}")

# Separar con 60 minutos de colchón
from datetime import timedelta
minuto_inicio_test = minuto_corte + timedelta(minutes=60)

df_train = df_ordenado.filter(F.col("minuto") <= minuto_corte)
df_test  = df_ordenado.filter(F.col("minuto") >= minuto_inicio_test)

n_tr = df_train.count()
n_te = df_test.count()

print(f"\nEntrenamiento : {n_tr:,} filas  (hasta {minuto_corte})")
print(f"Separación    : 60 minutos de colchón")
print(f"Prueba        : {n_te:,} filas  (desde {minuto_inicio_test})")

Total filas con rezagos: 3,622
Filas entrenamiento (80%): 2,897
Minuto de corte: 2026-11-19 23:33:00

Entrenamiento : 2,898 filas  (hasta 2026-11-19 23:33:00)
Separación    : 60 minutos de colchón
Prueba        : 665 filas  (desde 2026-11-20 00:33:00)


## Celda 5 — Media histórica por hora (entrenamiento)
---

In [5]:
# Media por hora calculada SOLO con entrenamiento (evita fuga de información)
df_media_hora = (
    df_train
    .withColumn("hora", F.hour("minuto"))
    .groupBy("hora")
    .agg(F.avg("temperatura_c").alias("media_hora"))
    .orderBy("hora")
)

print("Media histórica por hora del día (entrenamiento):")
df_media_hora.show(24, truncate=False)

# Guardar como JSON para el streaming
media_hora_dict = {
    str(row["hora"]): round(row["media_hora"], 4)
    for row in df_media_hora.collect()
}

with open(f"{MODELS}/media_por_hora.json", "w") as f:
    json.dump(media_hora_dict, f, indent=2)

print("✓ Guardado en models/wrangler_temperatura/media_por_hora.json")

Media histórica por hora del día (entrenamiento):
+----+------------------+
|hora|media_hora        |
+----+------------------+
|0   |20.84240692239858 |
|1   |19.767843668528865|
|2   |19.34370710784313 |
|3   |18.830845272904483|
|4   |18.3508059002433  |
|5   |17.862890186915887|
|6   |19.03754992492493 |
|7   |20.15269594801224 |
|8   |19.880267003367006|
|9   |20.894291959064322|
|10  |22.56670933641976 |
|11  |20.222767280219795|
|12  |21.754001385902033|
|13  |23.044023888888898|
|14  |23.671351065891475|
|15  |23.50891394389439 |
|16  |24.699287222222214|
|17  |24.85678513888889 |
|18  |23.265907500000004|
|19  |22.30139069444444 |
|20  |21.2409725        |
|21  |19.715106037414973|
|22  |19.72336199294533 |
|23  |20.914593660968674|
+----+------------------+

✓ Guardado en models/wrangler_temperatura/media_por_hora.json


## Celda 6 — Construir targets futuros
---

In [6]:
# Función para agregar el valor futuro como target
def agregar_target(df_base, df_ref, minutos):
    col_target = f"target_{minutos}m"
    df_fut = df_ref.select(
        F.col("minuto").alias(f"minuto_fut_{minutos}"),
        F.col("temperatura_c").alias(col_target)
    )
    return df_base.join(
        df_fut,
        df_base["minuto"] == (
            F.col(f"minuto_fut_{minutos}") - F.expr(f"INTERVAL {minutos} MINUTES")
        ),
        how="left"
    ).drop(f"minuto_fut_{minutos}")

df_targets = agregar_target(df_feat, df_feat, 15)
df_targets = agregar_target(df_targets, df_feat, 30)
df_targets = agregar_target(df_targets, df_feat, 60)

# Descartar filas sin target (los últimos minutos del dataset)
df_targets = df_targets.dropna(
    subset=["target_15m", "target_30m", "target_60m"]
)

print(f"Filas con targets completos: {df_targets.count():,}")
df_targets.select(
    "minuto", "temperatura_c",
    "target_15m", "target_30m", "target_60m"
).show(3, truncate=False)

Filas con targets completos: 2,782
+-------------------+-----------------+------------------+------------------+------------------+
|minuto             |temperatura_c    |target_15m        |target_30m        |target_60m        |
+-------------------+-----------------+------------------+------------------+------------------+
|2026-10-29 21:22:00|20.24215         |20.110633333333336|19.940225         |20.064275000000002|
|2026-11-01 10:50:00|18.903675        |19.0307           |19.310925         |19.900175         |
|2026-11-01 10:56:00|18.95183333333333|19.0848           |19.437350000000002|20.03115          |
+-------------------+-----------------+------------------+------------------+------------------+
only showing top 3 rows



## Celda 7 — Corte train/test sobre targets
---
es una técnica en aprendizaje automático que asegura que la distribución de la variable objetivo (target) se mantenga proporcionalmente igual en los conjuntos de entrenamiento y prueba. 

In [8]:
# Aplicar el mismo corte cronológico sobre df_targets
df_targets_ord = df_targets.orderBy("minuto")

df_train_t = df_targets_ord.filter(F.col("minuto") <= minuto_corte)
df_test_t  = df_targets_ord.filter(F.col("minuto") >= minuto_inicio_test)

n_tr_t = df_train_t.count()
n_te_t = df_test_t.count()

print(f"Entrenamiento con targets: {n_tr_t:,} filas")
print(f"Prueba con targets       : {n_te_t:,} filas")

# Unir media por hora al test para usarla como referencia
df_media_hora_spark = spark.createDataFrame(
    [(int(h), v) for h, v in media_hora_dict.items()],
    ["hora", "media_hora"]
)

df_test_con_media = df_test_t.withColumn(
    "hora", F.hour("minuto")
).join(df_media_hora_spark, on="hora", how="left")

print(f"\nTest con media por hora: {df_test_con_media.count():,} filas")

Entrenamiento con targets: 2,118 filas
Prueba con targets       : 605 filas

Test con media por hora: 605 filas


## Celda 8 — Predictores y modelos a 15 minutos
---

LinearRegression es un algoritmo de aprendizaje supervisado y machine learning que modela la relación lineal entre una variable dependiente y una o más variables independientes. 

Su objetivo principal es predecir valores numéricos continuos ajustando una línea recta (o hiperplano) a los datos mediante el método de mínimos cuadrados ordinarios. 

ElasticNet, Ridge: Ambos son variantes regularizadas de la regresión lineal que añaden una penalización a la función de costo para prevenir el sobreajuste (overfitting).


## Ridge
Regresión lineal con penalización L2. Reduce el peso de todas las variables para evitar sobreajuste, pero no elimina ninguna. Útil cuando todas las variables aportan información y se busca un modelo más estable.

## ElasticNet
Combina penalización L1 (tipo Lasso) y L2 (tipo Ridge). Permite tanto reducir pesos como eliminar variables que no aportan. Ideal cuando hay muchas variables, algunas correlacionadas y otras irrelevantes.

In [12]:
# Columnas predictoras (las mismas para los 3 horizontes)
FEATURES = [
    "temperatura_c", "temp_lag15", "temp_lag30", "temp_lag60",
    "delta_15", "humedad_pct", "presion_hpa", "hora_sin", "hora_cos"
]

assembler = VectorAssembler(inputCols=FEATURES, outputCol="features")

# Modelos candidatos
candidatos_15 = {
    "LinearRegression": LinearRegression(
        featuresCol="features", labelCol="target_15m", maxIter=100),
    "Ridge": LinearRegression(
        featuresCol="features", labelCol="target_15m",
        elasticNetParam=0.0, regParam=0.1),
    "ElasticNet": LinearRegression(
        featuresCol="features", labelCol="target_15m",
        elasticNetParam=0.5, regParam=0.1),
    "RandomForest": RandomForestRegressor(
        featuresCol="features", labelCol="target_15m",
        numTrees=100, seed=42),
}

evaluator_rmse = RegressionEvaluator(
    labelCol="target_15m", predictionCol="prediction", metricName="rmse")
evaluator_mae  = RegressionEvaluator(
    labelCol="target_15m", predictionCol="prediction", metricName="mae")
evaluator_r2   = RegressionEvaluator(
    labelCol="target_15m", predictionCol="prediction", metricName="r2")

# Preparar datos
df_tr_15 = assembler.transform(df_train_t)
df_te_15 = assembler.transform(df_test_con_media)

# Persistencia: predecir temperatura_c actual como predicción de t+15
rmse_pers_15 = df_te_15.select(
    F.sqrt(F.avg((F.col("temperatura_c") - F.col("target_15m"))**2)).alias("rmse")
).collect()[0]["rmse"]

mae_pers_15 = df_te_15.select(
    F.avg(F.abs(F.col("temperatura_c") - F.col("target_15m"))).alias("mae")
).collect()[0]["mae"]

print(f"Persistencia 15m — RMSE: {rmse_pers_15:.4f}  MAE: {mae_pers_15:.4f}")
print()

# Media histórica por hora como referencia
rmse_media_15 = df_te_15.select(
    F.sqrt(F.avg((F.col("media_hora") - F.col("target_15m"))**2)).alias("rmse")
).collect()[0]["rmse"]

mae_media_15 = df_te_15.select(
    F.avg(F.abs(F.col("media_hora") - F.col("target_15m"))).alias("mae")
).collect()[0]["mae"]

print(f"Media histórica 15m — RMSE: {rmse_media_15:.4f}  MAE: {mae_media_15:.4f}")
print()

# Entrenar y evaluar cada candidato
resultados_15 = {}
for nombre, modelo in candidatos_15.items():
    pipeline = Pipeline(stages=[modelo])
    fitted   = pipeline.fit(df_tr_15)
    preds    = fitted.transform(df_te_15)
    rmse = evaluator_rmse.evaluate(preds)
    mae  = evaluator_mae.evaluate(preds)
    r2   = evaluator_r2.evaluate(preds)
    resultados_15[nombre] = {"rmse": rmse, "mae": mae, "r2": r2, "model": fitted}
    print(f"{nombre:20s} — RMSE: {rmse:.4f}  MAE: {mae:.4f}  R²: {r2:.4f}")

#LinearRegression gana con RMSE 0,2052 — sí supera a la persistencia (0,2214)
#La media histórica es muy mala referencia a 15 min (RMSE 2,66)
#porque la temperatura cambia poco en 15 minutos y la hora del día
#no ayuda a ese horizonte corto

Persistencia 15m — RMSE: 0.2214  MAE: 0.1367

Media histórica 15m — RMSE: 2.6577  MAE: 2.5763

LinearRegression     — RMSE: 0.2052  MAE: 0.1113  R²: 0.9683
Ridge                — RMSE: 0.2057  MAE: 0.1442  R²: 0.9681
ElasticNet           — RMSE: 0.2097  MAE: 0.1502  R²: 0.9669
RandomForest         — RMSE: 0.3204  MAE: 0.2286  R²: 0.9227


## Celda 9 — Modelos a 30 minutos
---

In [13]:
# --- MODELOS A 30 MINUTOS ---
# Entrena y evalúa 4 candidatos contra persistencia y media histórica

evaluator_rmse_30 = RegressionEvaluator(
    labelCol="target_30m", predictionCol="prediction", metricName="rmse")
evaluator_mae_30  = RegressionEvaluator(
    labelCol="target_30m", predictionCol="prediction", metricName="mae")
evaluator_r2_30   = RegressionEvaluator(
    labelCol="target_30m", predictionCol="prediction", metricName="r2")

candidatos_30 = {
    "LinearRegression": LinearRegression(
        featuresCol="features", labelCol="target_30m", maxIter=100),
    "Ridge": LinearRegression(
        featuresCol="features", labelCol="target_30m",
        elasticNetParam=0.0, regParam=0.1),
    "ElasticNet": LinearRegression(
        featuresCol="features", labelCol="target_30m",
        elasticNetParam=0.5, regParam=0.1),
    "RandomForest": RandomForestRegressor(
        featuresCol="features", labelCol="target_30m",
        numTrees=100, seed=42),
}

# Persistencia a 30m
rmse_pers_30 = df_te_15.select(
    F.sqrt(F.avg((F.col("temperatura_c") - F.col("target_30m"))**2)).alias("rmse")
).collect()[0]["rmse"]
mae_pers_30 = df_te_15.select(
    F.avg(F.abs(F.col("temperatura_c") - F.col("target_30m"))).alias("mae")
).collect()[0]["mae"]
print(f"Persistencia 30m — RMSE: {rmse_pers_30:.4f}  MAE: {mae_pers_30:.4f}")

# Media histórica a 30m
rmse_media_30 = df_te_15.select(
    F.sqrt(F.avg((F.col("media_hora") - F.col("target_30m"))**2)).alias("rmse")
).collect()[0]["rmse"]
mae_media_30 = df_te_15.select(
    F.avg(F.abs(F.col("media_hora") - F.col("target_30m"))).alias("mae")
).collect()[0]["mae"]
print(f"Media histórica 30m — RMSE: {rmse_media_30:.4f}  MAE: {mae_media_30:.4f}")
print()

# Entrenar y evaluar
resultados_30 = {}
for nombre, modelo in candidatos_30.items():
    pipeline = Pipeline(stages=[modelo])
    fitted   = pipeline.fit(df_tr_15)
    preds    = fitted.transform(df_te_15)
    rmse = evaluator_rmse_30.evaluate(preds)
    mae  = evaluator_mae_30.evaluate(preds)
    r2   = evaluator_r2_30.evaluate(preds)
    resultados_30[nombre] = {"rmse": rmse, "mae": mae, "r2": r2, "model": fitted}
    print(f"{nombre:20s} — RMSE: {rmse:.4f}  MAE: {mae:.4f}  R²: {r2:.4f}")

Persistencia 30m — RMSE: 0.3584  MAE: 0.2489
Media histórica 30m — RMSE: 2.5529  MAE: 2.4745

LinearRegression     — RMSE: 0.3335  MAE: 0.2276  R²: 0.9170
Ridge                — RMSE: 0.3414  MAE: 0.2591  R²: 0.9130
ElasticNet           — RMSE: 0.3393  MAE: 0.2636  R²: 0.9141
RandomForest         — RMSE: 0.5960  MAE: 0.4963  R²: 0.7348


## Celda 10 — Modelos a 60 minutos
---
Mismo patrón, ahora apuntando a target_60m. Es el horizonte más difícil — aquí veremos si algún modelo sigue superando a la persistencia.

In [14]:
# --- MODELOS A 60 MINUTOS ---
# Horizonte más exigente: la temperatura puede cambiar más en una hora

evaluator_rmse_60 = RegressionEvaluator(
    labelCol="target_60m", predictionCol="prediction", metricName="rmse")
evaluator_mae_60  = RegressionEvaluator(
    labelCol="target_60m", predictionCol="prediction", metricName="mae")
evaluator_r2_60   = RegressionEvaluator(
    labelCol="target_60m", predictionCol="prediction", metricName="r2")

candidatos_60 = {
    "LinearRegression": LinearRegression(
        featuresCol="features", labelCol="target_60m", maxIter=100),
    "Ridge": LinearRegression(
        featuresCol="features", labelCol="target_60m",
        elasticNetParam=0.0, regParam=0.1),
    "ElasticNet": LinearRegression(
        featuresCol="features", labelCol="target_60m",
        elasticNetParam=0.5, regParam=0.1),
    "RandomForest": RandomForestRegressor(
        featuresCol="features", labelCol="target_60m",
        numTrees=100, seed=42),
}

# Persistencia a 60m
rmse_pers_60 = df_te_15.select(
    F.sqrt(F.avg((F.col("temperatura_c") - F.col("target_60m"))**2)).alias("rmse")
).collect()[0]["rmse"]
mae_pers_60 = df_te_15.select(
    F.avg(F.abs(F.col("temperatura_c") - F.col("target_60m"))).alias("mae")
).collect()[0]["mae"]
print(f"Persistencia 60m — RMSE: {rmse_pers_60:.4f}  MAE: {mae_pers_60:.4f}")

# Media histórica a 60m
rmse_media_60 = df_te_15.select(
    F.sqrt(F.avg((F.col("media_hora") - F.col("target_60m"))**2)).alias("rmse")
).collect()[0]["rmse"]
mae_media_60 = df_te_15.select(
    F.avg(F.abs(F.col("media_hora") - F.col("target_60m"))).alias("mae")
).collect()[0]["mae"]
print(f"Media histórica 60m — RMSE: {rmse_media_60:.4f}  MAE: {mae_media_60:.4f}")
print()

# Entrenar y evaluar
resultados_60 = {}
for nombre, modelo in candidatos_60.items():
    pipeline = Pipeline(stages=[modelo])
    fitted   = pipeline.fit(df_tr_15)
    preds    = fitted.transform(df_te_15)
    rmse = evaluator_rmse_60.evaluate(preds)
    mae  = evaluator_mae_60.evaluate(preds)
    r2   = evaluator_r2_60.evaluate(preds)
    resultados_60[nombre] = {"rmse": rmse, "mae": mae, "r2": r2, "model": fitted}
    print(f"{nombre:20s} — RMSE: {rmse:.4f}  MAE: {mae:.4f}  R²: {r2:.4f}")

Persistencia 60m — RMSE: 0.5347  MAE: 0.4446
Media histórica 60m — RMSE: 2.3436  MAE: 2.2723

LinearRegression     — RMSE: 0.4505  MAE: 0.3566  R²: 0.8507
Ridge                — RMSE: 0.4549  MAE: 0.3732  R²: 0.8478
ElasticNet           — RMSE: 0.4981  MAE: 0.4203  R²: 0.8174
RandomForest         — RMSE: 0.7681  MAE: 0.6710  R²: 0.5659


## Celda 11 — Tabla resumen de resultados
---
Consolida los RMSE, MAE y R² de todos los modelos y referencias en una sola tabla para incluir en el informe.

In [15]:
# --- TABLA RESUMEN DE TODOS LOS RESULTADOS ---
# Comparación completa: referencias vs modelos en los 3 horizontes

filas = []

# Referencias
filas.append(("Persistencia",    "referencia", 15, rmse_pers_15,  mae_pers_15,  None))
filas.append(("Media histórica", "referencia", 15, rmse_media_15, mae_media_15, None))
filas.append(("Persistencia",    "referencia", 30, rmse_pers_30,  mae_pers_30,  None))
filas.append(("Media histórica", "referencia", 30, rmse_media_30, mae_media_30, None))
filas.append(("Persistencia",    "referencia", 60, rmse_pers_60,  mae_pers_60,  None))
filas.append(("Media histórica", "referencia", 60, rmse_media_60, mae_media_60, None))

# Modelos
for nombre, res in resultados_15.items():
    filas.append((nombre, "modelo", 15, res["rmse"], res["mae"], res["r2"]))
for nombre, res in resultados_30.items():
    filas.append((nombre, "modelo", 30, res["rmse"], res["mae"], res["r2"]))
for nombre, res in resultados_60.items():
    filas.append((nombre, "modelo", 60, res["rmse"], res["mae"], res["r2"]))

df_resumen = spark.createDataFrame(
    filas,
    ["modelo", "tipo", "horizonte_min", "rmse", "mae", "r2"]
).orderBy("horizonte_min", "rmse")

print("TABLA RESUMEN — RMSE por horizonte (orden ascendente):")
df_resumen.show(30, truncate=False)

TABLA RESUMEN — RMSE por horizonte (orden ascendente):
+----------------+----------+-------------+-------------------+-------------------+------------------+
|modelo          |tipo      |horizonte_min|rmse               |mae                |r2                |
+----------------+----------+-------------+-------------------+-------------------+------------------+
|LinearRegression|modelo    |15           |0.20522167386991277|0.11131650322157738|0.9682897230983618|
|Ridge           |modelo    |15           |0.2057373275000235 |0.14424868769304458|0.9681301681932183|
|ElasticNet      |modelo    |15           |0.20967930499293536|0.15019464338685562|0.9668972007632138|
|Persistencia    |referencia|15           |0.22138433466067492|0.13667823691460054|NULL              |
|RandomForest    |modelo    |15           |0.3204376127373369 |0.22864378236843508|0.9226891303740867|
|Media histórica |referencia|15           |2.6576908486406023 |2.576276845730027  |NULL              |
|LinearRegression|

## Celda 12 — Guardar el modelo ganador
---
Guardamos el pipeline de LinearRegression a 60 minutos (el horizonte principal) junto con su JSON de métricas. También guarda los modelos de 15 y 30 min para el streaming.

In [16]:
# --- GUARDAR MODELOS GANADORES ---
# LinearRegression ganó en los 3 horizontes

# Guardar los 3 modelos (el streaming los necesita)
resultados_15["LinearRegression"]["model"].save(f"{MODELS}/lr_15m")
resultados_30["LinearRegression"]["model"].save(f"{MODELS}/lr_30m")
resultados_60["LinearRegression"]["model"].save(f"{MODELS}/lr_60m")

print("✓ Modelos guardados:")
print(f"  {MODELS}/lr_15m")
print(f"  {MODELS}/lr_30m")
print(f"  {MODELS}/lr_60m")

# Guardar métricas en JSON
metricas = {
    "modelo_ganador": "LinearRegression",
    "variable": "temperatura_c",
    "horizontes": {
        "15m": {
            "rmse_modelo":      round(resultados_15["LinearRegression"]["rmse"], 4),
            "mae_modelo":       round(resultados_15["LinearRegression"]["mae"],  4),
            "r2_modelo":        round(resultados_15["LinearRegression"]["r2"],   4),
            "rmse_persistencia":round(rmse_pers_15,  4),
            "rmse_media_hora":  round(rmse_media_15, 4),
            "gana_persistencia": resultados_15["LinearRegression"]["rmse"] < rmse_pers_15,
        },
        "30m": {
            "rmse_modelo":      round(resultados_30["LinearRegression"]["rmse"], 4),
            "mae_modelo":       round(resultados_30["LinearRegression"]["mae"],  4),
            "r2_modelo":        round(resultados_30["LinearRegression"]["r2"],   4),
            "rmse_persistencia":round(rmse_pers_30,  4),
            "rmse_media_hora":  round(rmse_media_30, 4),
            "gana_persistencia": resultados_30["LinearRegression"]["rmse"] < rmse_pers_30,
        },
        "60m": {
            "rmse_modelo":      round(resultados_60["LinearRegression"]["rmse"], 4),
            "mae_modelo":       round(resultados_60["LinearRegression"]["mae"],  4),
            "r2_modelo":        round(resultados_60["LinearRegression"]["r2"],   4),
            "rmse_persistencia":round(rmse_pers_60,  4),
            "rmse_media_hora":  round(rmse_media_60, 4),
            "gana_persistencia": resultados_60["LinearRegression"]["rmse"] < rmse_pers_60,
        },
    }
}

with open(f"{MODELS}/metricas.json", "w") as f:
    json.dump(metricas, f, indent=2, ensure_ascii=False)

print("\n✓ Métricas guardadas en models/wrangler_temperatura/metricas.json")

✓ Modelos guardados:
  /home/jovyan/work/models/wrangler_temperatura/lr_15m
  /home/jovyan/work/models/wrangler_temperatura/lr_30m
  /home/jovyan/work/models/wrangler_temperatura/lr_60m

✓ Métricas guardadas en models/wrangler_temperatura/metricas.json


## Celda 13 — Verificar que los modelos cargan bien
---
Carga cada modelo desde disco y hace una predicción de prueba para confirmar que el pipeline quedó bien serializado.

In [17]:
# --- VERIFICAR CARGA DE MODELOS ---
# Carga desde disco y hace una predicción de prueba con 3 filas

modelo_15 = PipelineModel.load(f"{MODELS}/lr_15m")
modelo_30 = PipelineModel.load(f"{MODELS}/lr_30m")
modelo_60 = PipelineModel.load(f"{MODELS}/lr_60m")

muestra = df_te_15.limit(3)

pred_15 = modelo_15.transform(muestra).select(
    "minuto", "temperatura_c", "target_15m",
    F.round("prediction", 4).alias("pred_15m")
)
pred_30 = modelo_30.transform(muestra).select(
    "minuto", F.round("prediction", 4).alias("pred_30m")
)
pred_60 = modelo_60.transform(muestra).select(
    "minuto", F.round("prediction", 4).alias("pred_60m")
)

df_verificacion = (
    pred_15
    .join(pred_30, on="minuto")
    .join(pred_60, on="minuto")
    .orderBy("minuto")
)

print("Verificación de carga — predicciones sobre 3 filas de prueba:")
df_verificacion.show(truncate=False)
print("✓ Los 3 modelos cargan y predicen correctamente")

Verificación de carga — predicciones sobre 3 filas de prueba:
+-------------------+-------------+------------------+--------+--------+--------+
|minuto             |temperatura_c|target_15m        |pred_15m|pred_30m|pred_60m|
+-------------------+-------------+------------------+--------+--------+--------+
|2026-11-19 21:25:00|23.92455     |23.83965          |23.7985 |23.5967 |23.316  |
|2026-11-20 00:08:00|22.7028      |22.833750000000002|22.5365 |22.2975 |21.9025 |
|2026-11-20 00:54:00|22.481925    |22.380625         |22.286  |22.0493 |21.639  |
+-------------------+-------------+------------------+--------+--------+--------+

✓ Los 3 modelos cargan y predicen correctamente


## Celda 14 — Verificar episodios sobre p95 en el conjunto de prueba
---
Antes de entrenar el clasificador, contamos cuántos episodios sobre p95 hay en prueba. Si son pocos o cero, la alerta no se puede evaluar bien.

In [18]:
# --- ALERTA: contar episodios sobre p95 en prueba ---
# Si no hay episodios, el clasificador no tiene casos positivos que aprender

df_alerta_check = (
    df_test_t
    .withColumn("sobre_p95_60m",
        F.when(F.col("target_60m") > P95, 1).otherwise(0)
    )
)

n_total_prueba   = df_alerta_check.count()
n_sobre_p95      = df_alerta_check.filter(F.col("sobre_p95_60m") == 1).count()
n_bajo_p5        = df_alerta_check.filter(F.col("target_60m") < P5).count()
pct_sobre        = n_sobre_p95 / n_total_prueba * 100

print(f"Conjunto de prueba: {n_total_prueba:,} filas")
print(f"Minutos futuros sobre p95 ({P95:.2f} °C): {n_sobre_p95:,}  ({pct_sobre:.1f}%)")
print(f"Minutos futuros bajo p5  ({P5:.2f} °C): {n_bajo_p5:,}")

if n_sobre_p95 == 0:
    print("\n⚠ No hay episodios sobre p95 en el conjunto de prueba.")
    print("  El clasificador de alerta no se puede evaluar — se omite.")
    ENTRENAR_ALERTA = False
elif n_sobre_p95 < 10:
    print(f"\n⚠ Solo {n_sobre_p95} episodios sobre p95. Muestra muy pequeña.")
    print("  Se entrena pero los resultados deben interpretarse con cautela.")
    ENTRENAR_ALERTA = True
else:
    print(f"\n✓ Suficientes episodios para entrenar el clasificador.")
    ENTRENAR_ALERTA = True

Conjunto de prueba: 605 filas
Minutos futuros sobre p95 (26.62 °C): 0  (0.0%)
Minutos futuros bajo p5  (16.53 °C): 0

⚠ No hay episodios sobre p95 en el conjunto de prueba.
  El clasificador de alerta no se puede evaluar — se omite.


## Celda 15 — Registrar resultado de alerta y cerrar notebook 02 
---
Se documenta que la alerta no se pudo evaluar por falta de episodios en el periodo de prueba, y genera el resumen final.

In [20]:
# --- RESULTADO DE ALERTA ---
# No hay episodios sobre p95 ni bajo p5 en el conjunto de prueba.
# Esto no es un error: el periodo de prueba (20 nov madrugada)
# tuvo temperaturas completamente dentro del rango normal.
# El clasificador se omite — se reporta tal cual en el informe.

alerta_resultado = {
    "evaluable": False,
    "motivo": "El conjunto de prueba no contiene episodios sobre p95 ni bajo p5",
    "n_prueba": n_total_prueba,
    "n_sobre_p95": n_sobre_p95,
    "n_bajo_p5": n_bajo_p5,
    "p95": P95,
    "p5": P5
}

with open(f"{MODELS}/alerta_resultado.json", "w") as f:
    json.dump(alerta_resultado, f, indent=2, ensure_ascii=False)

print("=" * 55)
print("RESUMEN NOTEBOOK 02 — WRANGLER TEMPERATURA")
print("=" * 55)
print(f"\nCorte cronológico:")
print(f"  Entrenamiento : {n_tr_t:,} filas (hasta {minuto_corte})")
print(f"  Prueba        : {n_te_t:,} filas (desde {minuto_inicio_test})")
print(f"\nModelo ganador en los 3 horizontes: LinearRegression")
print(f"\n  Horizonte | RMSE modelo | RMSE persist. | ¿Gana?")
print(f"  ----------|-------------|---------------|-------")
print(f"  15 min    |   {resultados_15['LinearRegression']['rmse']:.4f}    |    {rmse_pers_15:.4f}    |  {'✓ Sí' if resultados_15['LinearRegression']['rmse'] < rmse_pers_15 else '✗ No'}")
print(f"  30 min    |   {resultados_30['LinearRegression']['rmse']:.4f}    |    {rmse_pers_30:.4f}    |  {'✓ Sí' if resultados_30['LinearRegression']['rmse'] < rmse_pers_30 else '✗ No'}")
print(f"  60 min    |   {resultados_60['LinearRegression']['rmse']:.4f}    |    {rmse_pers_60:.4f}    |  {'✓ Sí' if resultados_60['LinearRegression']['rmse'] < rmse_pers_60 else '✗ No'}")
print(f"\nAlerta sobre p95: no evaluable (0 episodios en prueba)")
print(f"\nArchivos generados:")
print(f"  models/wrangler_temperatura/lr_15m/")
print(f"  models/wrangler_temperatura/lr_30m/")
print(f"  models/wrangler_temperatura/lr_60m/")
print(f"  models/wrangler_temperatura/metricas.json")
print(f"  models/wrangler_temperatura/media_por_hora.json")
print(f"  models/wrangler_temperatura/alerta_resultado.json")
print("\n✓ Notebook 02 completo")

RESUMEN NOTEBOOK 02 — WRANGLER TEMPERATURA

Corte cronológico:
  Entrenamiento : 2,118 filas (hasta 2026-11-19 23:33:00)
  Prueba        : 605 filas (desde 2026-11-20 00:33:00)

Modelo ganador en los 3 horizontes: LinearRegression

  Horizonte | RMSE modelo | RMSE persist. | ¿Gana?
  ----------|-------------|---------------|-------
  15 min    |   0.2052    |    0.2214    |  ✓ Sí
  30 min    |   0.3335    |    0.3584    |  ✓ Sí
  60 min    |   0.4505    |    0.5347    |  ✓ Sí

Alerta sobre p95: no evaluable (0 episodios en prueba)

Archivos generados:
  models/wrangler_temperatura/lr_15m/
  models/wrangler_temperatura/lr_30m/
  models/wrangler_temperatura/lr_60m/
  models/wrangler_temperatura/metricas.json
  models/wrangler_temperatura/media_por_hora.json
  models/wrangler_temperatura/alerta_resultado.json

✓ Notebook 02 completo
